In [ ]:
%%capture
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.preprocessing import LabelEncoder
from recsys_pipeliner.recommendations.transformer import (
    SimilarityTransformer,
    UserItemMatrixTransformer,
)
from recsys_pipeliner.recommendations.recommender import (
    SimilarityRecommender,
    ItemBasedRecommender,
    UserBasedRecommender,
)
from IPython.display import display

In [ ]:
# load test data
data_types = {"user_id": str, "item_id": str, "rating": np.float64}
user_item_ratings = pd.read_csv("../../tests/test_data/user_item_ratings_toy.csv", dtype=data_types)

display(user_item_ratings.head(3))

# encode the user/item ids
item_encoder = LabelEncoder()
user_encoder = LabelEncoder()

user_item_ratings["item_id"] = item_encoder.fit_transform(
    user_item_ratings["item_id"]
)
user_item_ratings["user_id"] = user_encoder.fit_transform(
    user_item_ratings["user_id"]
)

unique_users = pd.Series(user_encoder.classes_)
unique_items = pd.Series(item_encoder.classes_)

print("unique_users.shape", unique_users.shape)
print("unique_items.shape", unique_items.shape)

display(user_item_ratings.head(3))

# create the user/item matrix
user_item_matrix_transformer = UserItemMatrixTransformer()

user_item_matrix = user_item_matrix_transformer.transform(
    user_item_ratings.to_numpy(),
)

print("user_item_matrix.shape", user_item_matrix.shape)

# sanity check
users = user_item_ratings["user_id"].to_numpy().astype(int)
items = user_item_ratings["item_id"].to_numpy().astype(int)
ratings = user_item_ratings["rating"].to_numpy().astype(np.float32)
for user, item, rating in zip(users, items, ratings):
    assert user_item_matrix[user, item] == rating

In [ ]:
item_similarity_matrix_transformer = SimilarityTransformer()
item_similarity_matrix = item_similarity_matrix_transformer.transform(
    user_item_matrix.T
)

user_similarity_matrix_transformer = SimilarityTransformer()
user_similarity_matrix = user_similarity_matrix_transformer.transform(
    user_item_matrix
)

item_similarity_matrix.shape, user_similarity_matrix.shape

In [ ]:
# for use below
user_id = "U00003"
user_idx = user_encoder.transform([user_id])[0]
item_id = "I00003"
item_idx = item_encoder.transform([item_id])[0]
k=10

## Item-based prediction

Given a user_id and an item_id:

1. Get all items the user has rated
2. Sort by similarity to item_id
3. Get top k
4. Calculate weighted average rating
5. Return estimated score

In [ ]:
# manual implementation

_, single_users_rated_items, single_users_ratings = sp.sparse.find(user_item_matrix[user_idx, :])

# remove the item_idx from the users_rated_items
users_rated_items = single_users_rated_items[single_users_rated_items != item_idx]
users_ratings = single_users_ratings[single_users_rated_items != item_idx]

print("users_rated_items", item_encoder.inverse_transform(users_rated_items))
print("users_ratings", users_ratings)

# get the similarities to item_id
item_similarities = item_similarity_matrix[:, users_rated_items][item_idx].toarray().astype(np.float32).round(6)
print("item_similarities", item_similarities)

# sort by similarity (desc) and get top k
top_k_mask = np.argsort(1 - item_similarities)[:k]
print("top_k_mask", top_k_mask)

top_k_users_rated_items = users_rated_items[top_k_mask]
top_k_user_ratings = users_ratings[top_k_mask]
top_k_rated_item_similarities = item_similarities[top_k_mask]
users_unrated_items = np.setdiff1d(np.arange(item_similarity_matrix.shape[0]), top_k_users_rated_items)

print("top_k_users_rated_items", top_k_users_rated_items)
print("top_k_user_ratings", top_k_user_ratings)
print("top_k_rated_item_similarities", top_k_rated_item_similarities)

# weighted average rating
predicted_item_based_rating = np.average(top_k_user_ratings, axis=0, weights=top_k_rated_item_similarities).astype(np.float32).round(6)
print(f"predicted rating for item {item_id} by user {user_id}", predicted_item_based_rating)

In [ ]:
item_based_recommender = ItemBasedRecommender(k=k)
item_based_recommender.fit(user_item_matrix)

assert item_based_recommender.predict(user_idx, item_idx) == predicted_item_based_rating

for id in users_unrated_items[:10]:
    prediction = item_based_recommender.predict(user_idx, id)
    item_id = item_encoder.inverse_transform([id])[0]
    print(item_id, prediction)

## User-based prediction

1. Get users who have rated item_id
2. Sort by similarity to user_id
3. Get top k
4. Get those users' rating of item_id
5. Calculate weighted average rating
6. Return estimated score

In [ ]:
# manual implementation

_, all_users_with_ratings, all_users_ratings = sp.sparse.find(user_item_matrix[:, item_idx])

users = all_users_with_ratings[all_users_with_ratings != user_idx]
users_ratings = all_users_ratings[all_users_with_ratings != user_idx]

print("all_users_with_ratings", all_users_with_ratings)
print("all_users_ratings", all_users_ratings)

# get the similarities to user_id
_, similar_users, user_similarities = sp.sparse.find(user_similarity_matrix[user_idx, users])

print("similar_users", similar_users)
print("user_similarities", user_similarities)

user_similarities2 = user_similarity_matrix[user_idx, similar_users].toarray().astype(np.float32).round(6)
print("user_similarities2", user_similarities2)

# sort by similarity (desc) and get top k
top_k_mask = np.argsort(1 - user_similarities)[:k]
print("top_k_mask", top_k_mask)

top_k_users = users[top_k_mask]
top_k_users_ratings = users_ratings[top_k_mask]
top_k_users_similarities = user_similarities[top_k_mask]

print("top_k_users", top_k_users)
print("top_k_users_ratings", top_k_users_ratings)
print("top_k_users_similarities", top_k_users_similarities)

# weighted average rating
predicted_user_based_rating = np.average(top_k_users_ratings, axis=0, weights=top_k_users_similarities).astype(np.float32).round(6)
print(f"predicted rating for item {item_id} by user {user_id}", predicted_user_based_rating)

In [ ]:
user_based_recommender = UserBasedRecommender(k=k)
user_based_recommender.fit(user_item_matrix)

assert user_based_recommender.predict(user_idx, item_idx) == predicted_user_based_rating

for id in users_unrated_items[:10]:
    prediction = user_based_recommender.predict(user_idx, id)
    item_id = item_encoder.inverse_transform([id])[0]
    print(item_id, prediction)

In [ ]:
# Reimplemented using the new classes

